# HLCA — Manuscript-ready results (Table 1 + figures)

## Rationale (why this experiment exists)

The Human Lung Cell Atlas (HLCA) is a realistic integration setting where **identifier harmonization is a prerequisite** for atlas-scale model training.
This experiment is designed to market the *practical* value of IDTrack by turning harmonization into a **reportable, auditable artifact**:

- HLCA contains many studies/datasets that were processed under different Ensembl releases and annotation conventions.
- The same biological entity can appear under different identifiers, and conversely one identifier can legitimately expand to multiple candidates (splits/merges/history).
- IDTrack makes these outcomes explicit (1→0 / 1→1 / 1→n) and keeps the conversion reproducible via a snapshot-bounded graph.

## What this notebook produces (manuscript-facing)

- **Table 1 (canonical)**: `idtrack-manuscript/tables/hlca_harmonization.tex`
- **Extended diagnostics (optional)**: `idtrack-manuscript/tables/hlca_harmonization_extended.tex`
- **Alternative tables (optional)**:
  - `idtrack-manuscript/tables/hlca_harmonization_fractions.tex` (fractions; easier cross-dataset comparison)
  - `idtrack-manuscript/tables/hlca_feature_space_summary.tex` (union vs intersection; target comparison)
- **Figures (optional pool for the Results narrative)**:
  - `idtrack-manuscript/figures/fig_hlca_outcomes_hgnc.pdf`
  - `idtrack-manuscript/figures/fig_hlca_target_compare.pdf`
  - `idtrack-manuscript/figures/fig_hlca_gene_overlap.pdf`
  - `idtrack-manuscript/figures/fig_hlca_jaccard_heatmap.pdf`
  - `idtrack-manuscript/figures/fig_hlca_jaccard_heatmap_hgnc.pdf`
  - `idtrack-manuscript/figures/fig_hlca_unique_targets_compare.pdf`
  - `idtrack-manuscript/figures/fig_hlca_feature_space_union_intersection.pdf`
  - `idtrack-manuscript/figures/fig_hlca_throughput_scaling.pdf`
  - `idtrack-manuscript/figures/fig_hlca_marketing_suite.pdf` (multi-panel)

## Reproducibility contract (what readers can rerun)

This notebook intentionally foregrounds the knobs you can report in a Methods section:

- **Graph snapshot boundary** (what Ensembl history window is considered)
- **Target release** (what time point you harmonize into)
- **Target namespace** (HGNC vs Ensembl backbone; both are reported)
- **Ambiguity policy** (`strategy='all'` vs `strategy='best'`)

## Cache-first execution (no "RUN_*" toggles)

- If conversion caches exist, the notebook skips graph work and only composes tables/figures.
- If caches are missing, the notebook computes them once and writes under `idtrack/docs/_notebooks/idtrack_cache/experiments/hlca/`.

## Inputs and environment variables

- `HLCA_BASE_PATH`: HLCA data root (required only if caches are missing).
- `IDTRACK_LOCAL_REPO`: IDTrack cache directory (recommended: `idtrack/docs/_notebooks/idtrack_cache`).

This notebook uses the same curated HLCA study→files mapping as:
- `idtrack/docs/_notebooks/05_tutorial_harmonization.ipynb`

## Interpretation guide (how to read Table 1)

- **1→0**: no resolvable target (lost identifiers; measurable and reportable).
- **1→1**: unambiguous resolution.
- **1→n**: legitimate ambiguity (splits/merges/history); not a software error.
- **TDM vs ATM** (HGNC target): ATM indicates an Ensembl-side fallback where HGNC lacked a synonym for the target.


In [ ]:
from __future__ import annotations

import os
import time
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path (works even when launched from nested folders)
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    atomic_write_text,
    read_pickle,
    notebook_context,
    save_figure,
    safe_tag,
    write_pickle,
)

ctx = notebook_context('hlca', start=REPO_ROOT)
plt.rcParams.update({'savefig.dpi': 300})

IDTRACK_LOCAL_REPO = ctx.idtrack_local_repo
HLCA_CACHE = ctx.experiment_cache
MANUSCRIPT_TABLES = ctx.manuscript_tables
MANUSCRIPT_FIGURES = ctx.manuscript_figures

print('Repo root:', REPO_ROOT)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('HLCA_CACHE:', HLCA_CACHE)
print('MANUSCRIPT_TABLES:', MANUSCRIPT_TABLES)
print('MANUSCRIPT_FIGURES:', MANUSCRIPT_FIGURES)


In [ ]:
# -------------------- Configuration --------------------

# HLCA data root.
# Recommended: export in your shell so you never edit notebooks.
#   export HLCA_BASE_PATH=/path/to/HLCA_reproducibility/data
DEFAULT_HLCA_BASE_PATH = ''

base_path = os.environ.get('HLCA_BASE_PATH', DEFAULT_HLCA_BASE_PATH).strip()
HLCA_BASE_PATH = Path(base_path).expanduser().resolve() if base_path else None

# Shared IDTrack cache folder (graphs + databases + other heavy artefacts)
# and experiment cache are resolved in the setup cell via `notebook_context()`.
CONVERSIONS_DIR = (HLCA_CACHE / 'conversions')
CONVERSIONS_DIR.mkdir(parents=True, exist_ok=True)

# Manuscript choices
TARGET_RELEASE = 107
CONVERSION_STRATEGY = 'all'  # keep 'all' to expose 1→n explicitly

# Graph snapshot boundary used when conversions must be (re)computed.
# Leave as None to use the latest release reported by Ensembl REST.
GRAPH_SNAPSHOT_RELEASE: int | None = None

# Targets to evaluate.
# - `None` means: stay on Ensembl gene backbone (target IDs are Ensembl genes).
# - External DB names (e.g. 'HGNC Symbol') request final conversion into that namespace.
FINAL_DATABASES: dict[str, str | None] = {
    'ensembl_gene': None,
    'HGNC Symbol': 'HGNC Symbol',
}

print('HLCA_BASE_PATH:', HLCA_BASE_PATH)
print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('HLCA_CACHE:', HLCA_CACHE)
print('CONVERSIONS_DIR:', CONVERSIONS_DIR)
print('MANUSCRIPT_TABLES:', MANUSCRIPT_TABLES)
print('MANUSCRIPT_FIGURES:', MANUSCRIPT_FIGURES)


In [ ]:
# -------------------- Curated HLCA study→files mapping --------------------

# Mirrors `idtrack/docs/_notebooks/05_tutorial_harmonization.ipynb`.
# If your HLCA directory structure differs, edit the two dataset directories below.

if HLCA_BASE_PATH is None:
    print('Set HLCA_BASE_PATH to your HLCA data root to enable this notebook.')

DEFAULT_DSET0_REL = 'HLCA_extended/extension_datasets/ready/full'
DEFAULT_DSET1_REL = 'HLCA_extended/extension_datasets/raw'

# These reflect the standard HLCA reproducibility layout.
dset0_dir = (HLCA_BASE_PATH / DEFAULT_DSET0_REL) if HLCA_BASE_PATH else None
dset1_dir = (HLCA_BASE_PATH / DEFAULT_DSET1_REL) if HLCA_BASE_PATH else None

hlca_adata_dict: dict[str, list[str]] = {}

if dset0_dir and dset1_dir:
    hlca_adata_dict = {
        'Kaminski_2020': [f'{dset0_dir}/adams.h5ad'],
        'Meyer_2021': [f'{dset0_dir}/meyer_2021.h5ad'],
        'MeyerNikolic_unpubl': [f'{dset0_dir}/meyer_nikolic_unpubl.h5ad'],
        'Barbry_unpubl': [f'{dset0_dir}/barbry.h5ad'],
        'Regev_2021': [
            f'{dset0_dir}/delorey_cryo.h5ad',
            f'{dset0_dir}/delorey_fresh.h5ad',
            f'{dset0_dir}/delorey_nuclei.h5ad',
        ],
        'Thienpont_2018': [f'{dset1_dir}/Lambrechts/lambrechts.h5ad'],
        'Budinger_2020': [f'{dset0_dir}/bharat.h5ad'],
        'Banovich_Kropski_2020': [f'{dset0_dir}/haberman.h5ad'],
        'Sheppard_2020': [f'{dset0_dir}/tsukui.h5ad'],
        'Wunderink_2021': [f'{dset0_dir}/grant_cryo.h5ad', f'{dset0_dir}/grant_fresh.h5ad'],
        'Lambrechts_2021': [f'{dset0_dir}/wouters.h5ad'],
        'Zhang_2021': [f'{dset1_dir}/Liao/covid_for_publish.h5ad'],
        'Duong_lungMAP_unpubl': [f'{dset0_dir}/duong.h5ad'],
        'Janssen_2020': [f'{dset0_dir}/mould.h5ad'],
        'Sun_2020': [
            f'{dset0_dir}/wang_sub_batch1.h5ad',
            f'{dset0_dir}/wang_sub_batch2.h5ad',
            f'{dset0_dir}/wang_sub_batch3.h5ad',
            f'{dset0_dir}/wang_sub_batch4.h5ad',
        ],
        'Gomperts_2021': [
            f'{dset0_dir}/carraro_ucla.h5ad',
            f'{dset0_dir}/carraro_cff.h5ad',
            f'{dset0_dir}/carraro_csmc.h5ad',
        ],
        'Eils_2020': [f'{dset0_dir}/lukassen.h5ad'],
        'Schiller_2020': [f'{dset0_dir}/mayr.h5ad'],
        'Misharin_Budinger_2018': [f'{dset0_dir}/reyfman_disease.h5ad'],
        'Shalek_2018': [f'{dset0_dir}/ordovasmontanes.h5ad'],
        'Schiller_2021': [f'{dset0_dir}/schiller_discovair.h5ad'],
        'Peer_Massague_2020': [f'{dset0_dir}/laughney.h5ad'],
        'Lafyatis_2019': [f'{dset0_dir}/valenzi.h5ad'],
        'Tata_unpubl': [f'{dset0_dir}/tata_unpubl.h5ad'],
        'Xu_2020': [f'{dset0_dir}/guo.h5ad'],
        'Sims_2019': [f'{dset0_dir}/szabo.h5ad'],
        'Schultze_unpubl': [f'{dset0_dir}/schultze.h5ad'],
    }

print('Studies configured:', len(hlca_adata_dict))
print('Example:', list(hlca_adata_dict)[:5])


In [ ]:
# -------------------- Validate file availability --------------------

flat_rows: list[tuple[str, str, bool]] = []
missing_rows: list[tuple[str, str]] = []

for study, paths in hlca_adata_dict.items():
    for p in paths:
        exists = Path(p).exists()
        flat_rows.append((study, p, exists))
        if not exists:
            missing_rows.append((study, p))

hlca_file_status = pd.DataFrame(flat_rows, columns=['study', 'path', 'exists'])

if hlca_file_status.empty:
    print('No HLCA file mapping available (HLCA_BASE_PATH unset or directory layout not found).')
else:
    print('Total referenced .h5ad files:', int(hlca_file_status.shape[0]))
    print('Total found on disk:', int(hlca_file_status['exists'].sum()))

    if missing_rows:
        print()
        print(f"Missing files referenced by the curated list: {len(missing_rows)}")
        print('Showing first 15 missing:')
        for study, p in missing_rows[:15]:
            print(f' - {study}: {p}')

if hlca_file_status.empty:
    hlca_file_status
else:
    (
        hlca_file_status.groupby('study')['exists']
        .agg(['count', 'sum'])
        .rename(columns={'count': 'n_files', 'sum': 'n_found'})
        .sort_values(['n_found', 'n_files'], ascending=False)
    )


In [ ]:
# -------------------- Conversion cache I/O --------------------


def _safe_stem(s: str) -> str:
    return ''.join(c if c.isalnum() or c in {'-', '_'} else '_' for c in str(s))


def _final_label(final_database: str | None) -> str:
    return 'ensembl_gene' if final_database is None else str(final_database)


def _pickle_path(study: str, *, final_database: str | None) -> Path:
    tag = (
        f"hlca_{_safe_stem(study)}_toRelease{TARGET_RELEASE}_final{_safe_stem(_final_label(final_database))}"
        f"_strategy{CONVERSION_STRATEGY}.pickle"
    )
    return CONVERSIONS_DIR / tag


@dataclass(frozen=True)
class ConversionPayload:
    matchings: list[dict]
    seconds: float
    n_inputs: int
    graph_snapshot_release: int | None
    target_release: int
    final_database: str | None
    strategy: str

    @property
    def it_per_s(self) -> float:
        return (self.n_inputs / self.seconds) if self.seconds else float('nan')


def _load_payload(study: str, *, final_database: str | None) -> ConversionPayload | None:
    p = _pickle_path(study, final_database=final_database)
    if not p.exists():
        return None

    obj = read_pickle(p)

    # Backwards-compat: some earlier caches stored just `list[dict]`.
    if isinstance(obj, list):
        return ConversionPayload(
            matchings=obj,
            seconds=float('nan'),
            n_inputs=len(obj),
            graph_snapshot_release=None,
            target_release=TARGET_RELEASE,
            final_database=final_database,
            strategy=CONVERSION_STRATEGY,
        )

    if not isinstance(obj, dict) or 'matchings' not in obj:
        raise TypeError(f"Unexpected payload in {p}: expected dict with 'matchings'.")

    matchings = obj['matchings']
    return ConversionPayload(
        matchings=matchings,
        seconds=float(obj.get('seconds', float('nan'))),
        n_inputs=int(obj.get('n_inputs', len(matchings))),
        graph_snapshot_release=(int(obj['graph_snapshot_release']) if obj.get('graph_snapshot_release') is not None else None),
        target_release=int(obj.get('target_release', TARGET_RELEASE)),
        final_database=obj.get('final_database', final_database),
        strategy=str(obj.get('strategy', CONVERSION_STRATEGY)),
    )


def _save_payload(study: str, *, final_database: str | None, payload: ConversionPayload) -> Path:
    p = _pickle_path(study, final_database=final_database)
    return write_pickle(
        {
            'matchings': payload.matchings,
            'seconds': payload.seconds,
            'n_inputs': payload.n_inputs,
            'graph_snapshot_release': payload.graph_snapshot_release,
            'target_release': payload.target_release,
            'final_database': payload.final_database,
            'strategy': payload.strategy,
        },
        p,
    )


In [ ]:
# -------------------- Ensure conversion caches exist --------------------


def _studies_with_cached_conversions() -> set[str]:
    studies = set()
    for p in CONVERSIONS_DIR.glob('hlca_*_toRelease*_final*_strategy*.pickle'):
        name = p.name
        if not name.startswith('hlca_'):
            continue
        # hlca_<study>_toRelease...
        mid = name[len('hlca_'):]
        study = mid.split('_toRelease', 1)[0]
        if study:
            studies.add(study)
    return studies


def _studies_with_h5ad_files() -> list[str]:
    if hlca_file_status.empty:
        return []
    return [s for s, grp in hlca_file_status.groupby('study') if grp['exists'].any()]


def _read_var_names_backed(h5ad_path: str) -> list[str]:
    import anndata as ad

    adata = ad.read_h5ad(h5ad_path, backed='r')
    try:
        return adata.var_names.astype(str).tolist()
    finally:
        try:
            adata.file.close()
        except Exception:
            pass


def _study_union_gene_list(study: str) -> list[str]:
    genes: set[str] = set()
    for p in hlca_adata_dict.get(study, []):
        if Path(p).exists():
            genes.update(_read_var_names_backed(p))
    return sorted(genes)


# Decide which studies to process.
# - If HLCA paths are available, we use the curated mapping.
# - Otherwise, we rely on whatever is already cached on disk.
studies = _studies_with_h5ad_files() or sorted(_studies_with_cached_conversions())

missing_jobs: list[tuple[str, str, str | None]] = []
for study in studies:
    for label, final_db in FINAL_DATABASES.items():
        if _load_payload(study, final_database=final_db) is None:
            missing_jobs.append((study, label, final_db))

if not studies:
    print('No HLCA studies detected (no HLCA paths and no cached conversions).')
elif not missing_jobs:
    print(f'All conversion caches present for {len(studies)} studies; skipping IDTrack graph load.')
else:
    if not hlca_adata_dict:
        raise RuntimeError(
            'Conversion caches are missing, but HLCA_BASE_PATH is not configured. '            'Set HLCA_BASE_PATH so the notebook can read HLCA .h5ad var_names and compute caches.'
        )

    try:
        import idtrack
    except ImportError as e:
        raise ImportError('Computing HLCA caches requires `idtrack` installed.') from e

    # Build graph once.
    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    organism, latest_release = api.resolve_organism('human')
    snapshot = int(GRAPH_SNAPSHOT_RELEASE) if GRAPH_SNAPSHOT_RELEASE is not None else int(latest_release)
    snapshot = max(snapshot, int(TARGET_RELEASE))

    print(f'Building/loading IDTrack graph for {organism} snapshot_release={snapshot} (target_release={TARGET_RELEASE})')
    api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)

    # Cache union gene lists per study to avoid re-reading .h5ad files.
    gene_lists: dict[str, list[str]] = {}

    runs = []
    for study, label, final_db in missing_jobs:
        if study not in gene_lists:
            gene_lists[study] = _study_union_gene_list(study)

        ids = gene_lists[study]
        if not ids:
            print('Skip (no genes found):', study)
            continue

        t0 = time.perf_counter()
        matchings = api.convert_identifier_multiple(
            ids,
            to_release=int(TARGET_RELEASE),
            final_database=final_db,
            strategy=CONVERSION_STRATEGY,
            verbose=True,
            pbar_prefix=f"HLCA:{study}:{label}",
        )
        dt = time.perf_counter() - t0

        payload = ConversionPayload(
            matchings=matchings,
            seconds=dt,
            n_inputs=len(ids),
            graph_snapshot_release=snapshot,
            target_release=int(TARGET_RELEASE),
            final_database=final_db,
            strategy=CONVERSION_STRATEGY,
        )

        out_p = _save_payload(study, final_database=final_db, payload=payload)
        runs.append(
            {
                'study': study,
                'label': label,
                'final_database': final_db if final_db is not None else 'ensembl_gene',
                'n_inputs': payload.n_inputs,
                'seconds': payload.seconds,
                'it_per_s': payload.it_per_s,
                'graph_snapshot_release': snapshot,
                'target_release': int(TARGET_RELEASE),
                'strategy': CONVERSION_STRATEGY,
            }
        )
        print('Saved:', out_p)

    if runs:
        timings = pd.DataFrame(runs).sort_values(['label', 'study']).reset_index(drop=True)
        timings_path = HLCA_CACHE / 'timings.csv'
        atomic_write_text(timings_path, timings.to_csv(index=False))
        print('Wrote timings:', timings_path)
        timings
    else:
        print('No caches were written (nothing to run).')


In [ ]:
# -------------------- Build Table 1 (from cached conversions) --------------------

from collections import Counter

try:
    import idtrack
except ImportError as e:
    raise ImportError('This analysis requires `idtrack` installed. Graph loading is only needed if caches are missing.') from e

api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))

# Load timings if available
_timings_path = HLCA_CACHE / 'timings.csv'
if _timings_path.exists():
    timings = pd.read_csv(_timings_path)
else:
    timings = pd.DataFrame(columns=['study', 'label', 'it_per_s'])


def _timing_lookup(study: str, label: str) -> float:
    if timings.empty:
        return float('nan')
    sub = timings[(timings['study'] == study) & (timings['label'] == label)]
    if sub.empty:
        return float('nan')
    return float(sub.iloc[0]['it_per_s'])


def _summarize_matchings(matchings: list[dict]) -> dict[str, int]:
    bins = api.classify_multiple_conversion(matchings)

    # 1→0 / 1→1 / 1→n
    n_inputs = len(bins['input_identifiers'])
    n_1to0 = len(bins['matching_1_to_0'])
    n_1to1 = len(bins['matching_1_to_1'])
    n_1ton = len(bins['matching_1_to_n'])

    # Diagnostics
    n_changed_1to1 = len(bins['changed_only_1_to_1'])
    n_changed_1ton = len(bins['changed_only_1_to_n'])
    n_fallback_1to1 = len(bins['alternative_target_1_to_1'])
    n_fallback_1ton = len(bins['alternative_target_1_to_n'])

    # n→1 collapses among 1→1 targets (important for harmonization)
    t1 = []
    for rec in bins['matching_1_to_1']:
        tid = rec.get('target_id', [])
        if isinstance(tid, list) and len(tid) == 1:
            t1.append(str(tid[0]))
    n_to_1_targets = sum(1 for _k, v in Counter(t1).items() if v > 1)

    return {
        'input_ids': n_inputs,
        'one_to_none': n_1to0,
        'one_to_one': n_1to1,
        'one_to_many': n_1ton,
        'changed_only_1_to_1': n_changed_1to1,
        'changed_only_1_to_n': n_changed_1ton,
        'fallback_1_to_1': n_fallback_1to1,
        'fallback_1_to_n': n_fallback_1ton,
        'n_to_1_targets_within_1_to_1': n_to_1_targets,
    }


def _targets_1to1(matchings: list[dict]) -> set[str]:
    out = set()
    bins = api.classify_multiple_conversion(matchings)
    for rec in bins['matching_1_to_1']:
        tid = rec.get('target_id', [])
        if isinstance(tid, list) and len(tid) == 1:
            out.add(str(tid[0]))
    return out


def _targets_1to1_including_fallback(matchings: list[dict]) -> set[str]:
    """Return the set of unique 1→1 targets, including alternative-target fallbacks.

    This is the most relevant definition for harmonization because fallbacks still contribute
    stable 1→1 targets in the *final namespace*.
    """
    out = set()
    bins = api.classify_multiple_conversion(matchings)

    for rec in (bins['matching_1_to_1'] + bins['alternative_target_1_to_1']):
        tid = rec.get('target_id', [])
        if isinstance(tid, list) and len(tid) == 1:
            out.add(str(tid[0]))

    return out


def _n_to_1_targets_within_1to1(matchings: list[dict], *, include_fallback: bool = True) -> int:
    """Count how many *targets* are hit by multiple 1→1 queries (n→1 collisions)."""
    bins = api.classify_multiple_conversion(matchings)
    recs = list(bins['matching_1_to_1'])
    if include_fallback:
        recs += list(bins['alternative_target_1_to_1'])

    targets = []
    for rec in recs:
        tid = rec.get('target_id', [])
        if isinstance(tid, list) and len(tid) == 1:
            targets.append(str(tid[0]))

    return int(sum(1 for _k, v in Counter(targets).items() if v > 1))


# Determine studies from either curated HLCA mapping or cached conversions.
if hlca_file_status.empty:
    studies = sorted({s for s in _studies_with_cached_conversions()})
else:
    studies = [s for s, grp in hlca_file_status.groupby('study') if grp['exists'].any()]

rows = []
missing = []

# For feature-space diagnostics (use 1→1 target sets)
per_study_ensembl_targets_1to1: dict[str, set[str]] = {}
per_study_hgnc_targets_1to1: dict[str, set[str]] = {}

for study in studies:
    p_hgnc = _load_payload(study, final_database='HGNC Symbol')
    p_ens = _load_payload(study, final_database=None)

    if p_hgnc is None or p_ens is None:
        missing.append(study)
        continue

    c_hgnc = _summarize_matchings(p_hgnc.matchings)
    c_ens = _summarize_matchings(p_ens.matchings)

    if c_hgnc['one_to_none'] != c_ens['one_to_none']:
        print('Warning: 1→0 differs between targets for', study, c_hgnc['one_to_none'], c_ens['one_to_none'])

    per_study_ensembl_targets_1to1[study] = _targets_1to1(p_ens.matchings)
    per_study_hgnc_targets_1to1[study] = _targets_1to1_including_fallback(p_hgnc.matchings)

    # Harmonization-relevant diagnostics: how many *unique* 1→1 targets you get (feature-space size)
    ensembl_unique_targets = len(per_study_ensembl_targets_1to1[study])
    hgnc_unique_targets = len(per_study_hgnc_targets_1to1[study])

    # How many target IDs are hit by multiple 1→1 queries (n→1 collisions) in the final namespace
    hgnc_n_to_1_targets = _n_to_1_targets_within_1to1(p_hgnc.matchings, include_fallback=True)

    rows.append(
        {
            'Dataset': study,
            'Input IDs': c_ens['input_ids'],
            # HGNC target (TDM vs ATM = fallback)
            'HGNC 1→1 TDM': c_hgnc['one_to_one'],
            'HGNC 1→1 ATM': c_hgnc['fallback_1_to_1'],
            'HGNC 1→n TDM': c_hgnc['one_to_many'],
            'HGNC 1→n ATM': c_hgnc['fallback_1_to_n'],
            # Ensembl target (stay on backbone)
            'Ensembl 1→1': c_ens['one_to_one'],
            'Ensembl 1→n': c_ens['one_to_many'],
            '1→0': c_ens['one_to_none'],
            # Extra diagnostics (optional tables/figures)
            'Changed-only 1→1 (Ensembl)': c_ens['changed_only_1_to_1'],
            'Changed-only 1→n (Ensembl)': c_ens['changed_only_1_to_n'],
            'n→1 targets within 1→1 (Ensembl)': c_ens['n_to_1_targets_within_1_to_1'],
            'Unique 1→1 targets (Ensembl)': ensembl_unique_targets,
            'Unique 1→1 targets (HGNC, incl fallback)': hgnc_unique_targets,
            'n→1 targets within 1→1 (HGNC, incl fallback)': hgnc_n_to_1_targets,
            # Speed
            'Performance (it/s) Ensembl IDs': _timing_lookup(study, 'ensembl_gene'),
            'Performance (it/s) HGNC Symbols': _timing_lookup(study, 'HGNC Symbol'),
        }
    )

if missing:
    print('Missing cached conversions for studies:', missing)

summary = pd.DataFrame(rows).sort_values('Dataset').reset_index(drop=True)

summary_path = HLCA_CACHE / 'hlca_summary.csv'
atomic_write_text(summary_path, summary.to_csv(index=False))
print('Wrote:', summary_path)

summary

# -------------------- Cross-dataset overlap diagnostics (Ensembl 1→1 only) --------------------

ens_union_genes: int | None = None
ens_core_genes: int | None = None
ens_n_datasets: int | None = None

if per_study_ensembl_targets_1to1:
    counter = Counter()
    for s, genes in per_study_ensembl_targets_1to1.items():
        for g in genes:
            counter[g] += 1

    n_datasets = len(per_study_ensembl_targets_1to1)
    overlap = Counter(counter.values())

    overlap_df = (
        pd.DataFrame({'n_datasets_present': list(overlap.keys()), 'n_genes': list(overlap.values())})
        .sort_values('n_datasets_present')
        .reset_index(drop=True)
    )

    overlap_path = HLCA_CACHE / 'hlca_gene_overlap_distribution.csv'
    atomic_write_text(overlap_path, overlap_df.to_csv(index=False))
    print('Wrote:', overlap_path)

    core = int(overlap.get(n_datasets, 0))
    union = int(sum(overlap.values()))
    ens_union_genes = union
    ens_core_genes = core
    ens_n_datasets = n_datasets
    print(f'Ensembl (1→1 only) union genes: {union:,}')
    print(f'Ensembl (1→1 only) intersection genes (present in all {n_datasets} datasets): {core:,}')

    overlap_df
else:
    print('No per-study Ensembl 1→1 targets available; overlap diagnostics skipped.')

# -------------------- Pairwise overlap matrix (Ensembl 1→1 only) --------------------

# This supports a manuscript-friendly heatmap that markets "shared feature-space structure".
# The computation is cacheable and uses only the per-study 1→1 target sets extracted above.

if per_study_ensembl_targets_1to1:
    studies = sorted(per_study_ensembl_targets_1to1)
    jaccard = pd.DataFrame(index=studies, columns=studies, dtype=float)

    for s1 in studies:
        a = set(per_study_ensembl_targets_1to1[s1])
        for s2 in studies:
            b = set(per_study_ensembl_targets_1to1[s2])
            denom = len(a | b)
            jaccard.loc[s1, s2] = (len(a & b) / denom) if denom else float('nan')

    jaccard_path = HLCA_CACHE / 'hlca_jaccard_matrix.csv'
    atomic_write_text(jaccard_path, jaccard.to_csv())
    print('Wrote:', jaccard_path)

# -------------------- Cross-dataset overlap diagnostics (HGNC 1→1 only; incl fallback) --------------------

hgnc_union_genes: int | None = None
hgnc_core_genes: int | None = None
hgnc_n_datasets: int | None = None

if per_study_hgnc_targets_1to1:
    counter = Counter()
    for s, genes in per_study_hgnc_targets_1to1.items():
        for g in genes:
            counter[g] += 1

    n_datasets = len(per_study_hgnc_targets_1to1)
    overlap = Counter(counter.values())

    overlap_df = (
        pd.DataFrame({'n_datasets_present': list(overlap.keys()), 'n_genes': list(overlap.values())})
        .sort_values('n_datasets_present')
        .reset_index(drop=True)
    )

    overlap_path = HLCA_CACHE / 'hlca_hgnc_gene_overlap_distribution.csv'
    atomic_write_text(overlap_path, overlap_df.to_csv(index=False))
    print('Wrote:', overlap_path)

    core = int(overlap.get(n_datasets, 0))
    union = int(sum(overlap.values()))
    hgnc_union_genes = union
    hgnc_core_genes = core
    hgnc_n_datasets = n_datasets
    print(f'HGNC (1→1 incl fallback) union genes: {union:,}')
    print(f'HGNC (1→1 incl fallback) intersection genes (present in all {n_datasets} datasets): {core:,}')

    overlap_df
else:
    print('No per-study HGNC 1→1 targets available; overlap diagnostics skipped.')

# -------------------- Pairwise overlap matrix (HGNC 1→1 only; incl fallback) --------------------

if per_study_hgnc_targets_1to1:
    studies = sorted(per_study_hgnc_targets_1to1)
    jaccard = pd.DataFrame(index=studies, columns=studies, dtype=float)

    for s1 in studies:
        a = set(per_study_hgnc_targets_1to1[s1])
        for s2 in studies:
            b = set(per_study_hgnc_targets_1to1[s2])
            denom = len(a | b)
            jaccard.loc[s1, s2] = (len(a & b) / denom) if denom else float('nan')

    jaccard_path = HLCA_CACHE / 'hlca_hgnc_jaccard_matrix.csv'
    atomic_write_text(jaccard_path, jaccard.to_csv())
    print('Wrote:', jaccard_path)

# -------------------- Feature-space summary (Ensembl vs HGNC; 1→1 only) --------------------

feature_rows = []
if ens_union_genes is not None:
    feature_rows.append(
        {
            'target': 'Ensembl (1→1 only)',
            'n_datasets': int(ens_n_datasets or 0),
            'union_genes': int(ens_union_genes),
            'intersection_genes': int(ens_core_genes or 0),
        }
    )
if hgnc_union_genes is not None:
    feature_rows.append(
        {
            'target': 'HGNC (1→1 incl fallback)',
            'n_datasets': int(hgnc_n_datasets or 0),
            'union_genes': int(hgnc_union_genes),
            'intersection_genes': int(hgnc_core_genes or 0),
        }
    )

if feature_rows:
    fs = pd.DataFrame(feature_rows)
    fs_path = HLCA_CACHE / 'hlca_feature_space_summary.csv'
    atomic_write_text(fs_path, fs.to_csv(index=False))
    print('Wrote:', fs_path)


In [ ]:
# -------------------- Export LaTeX tables --------------------

out_tex_main = MANUSCRIPT_TABLES / 'hlca_harmonization.tex'
out_tex_ext = MANUSCRIPT_TABLES / 'hlca_harmonization_extended.tex'


def _latex_escape(value: str) -> str:
    s = str(value)
    s = s.replace('\\', r'\textbackslash{}')
    s = s.replace('&', r'\&')
    s = s.replace('%', r'\%')
    s = s.replace('_', r'\_')
    s = s.replace('#', r'\#')
    s = s.replace('{', r'\{')
    s = s.replace('}', r'\}')
    s = s.replace('^', r'\textasciicircum{}')
    s = s.replace('~', r'\textasciitilde{}')
    return s


def _fmt_int(x) -> str:
    try:
        return f"{int(x)}"
    except Exception:
        return str(x)


def _fmt_float(x, digits: int = 2) -> str:
    try:
        v = float(x)
        return f"{v:.{digits}f}" if v == v else ''
    except Exception:
        return ''


if summary.empty:
    print('No rows to export. Are HLCA conversions cached?')
else:
    caption = (
        'Summary of Human Lung Cell Atlas (HLCA) identifier harmonization performed with \textit{IDTrack}. '
        'All datasets are mapped into Ensembl release 107. The table reports explicit 1→0 / 1→1 / 1→n outcomes '
        'and distinguishes target-database matches from Ensembl fallbacks when the target database lacks a synonym.'
    )

    cols_main = [
        'Dataset',
        'Input IDs',
        'HGNC 1→1 TDM',
        'HGNC 1→1 ATM',
        'HGNC 1→n TDM',
        'HGNC 1→n ATM',
        'Ensembl 1→1',
        'Ensembl 1→n',
        '1→0',
        'Performance (it/s) Ensembl IDs',
        'Performance (it/s) HGNC Symbols',
    ]

    lines = []
    lines.append(r'\begin{table*}[t]')
    lines.append(r'\caption{')
    lines.append('    ' + caption)
    lines.append(r'}')
    lines.append(r'\label{tab:hlca}')
    lines.append(r'\centering')
    lines.append(r'\footnotesize')
    lines.append(r'\setlength{\tabcolsep}{3pt}')
    lines.append(r'\renewcommand{\arraystretch}{1.15}')
    lines.append(r'\begin{tabular}{@{}l r r r r r r r r r r@{}}')
    lines.append(r'\toprule')
    lines.append(r'& & \multicolumn{4}{c}{Target HGNC Symbols} & \multicolumn{2}{c}{Target Ensembl genes} & & \multicolumn{2}{c}{Performance (\textit{it/s})} \\')
    lines.append(r'\cmidrule(lr){3-6} \cmidrule(lr){7-8} \cmidrule(lr){10-11}')
    lines.append(
        r'Dataset & \shortstack[c]{Input\\IDs} & \multicolumn{2}{c}{\shortstack[c]{One-to-\\One}} & '
        r'\multicolumn{2}{c}{\shortstack[c]{One-to-\\Many}} & '
        r'\shortstack[c]{One-to-\\One} & \shortstack[c]{One-to-\\Many} & '
        r'\shortstack[c]{One-to-\\None} & '
        r'\shortstack[c]{Ensembl\\IDs} & \shortstack[c]{HGNC\\Symbols} \\'
    )
    lines.append(r'\cmidrule(lr){3-4} \cmidrule(lr){5-6}')
    lines.append(r'& & TDM & ATM & TDM & ATM \\')
    lines.append(r'\midrule')

    for _, r in summary[cols_main].iterrows():
        ds = _latex_escape(r['Dataset'])
        row = [
            f"\\textit{{{ds}}}",
            _fmt_int(r['Input IDs']),
            _fmt_int(r['HGNC 1→1 TDM']),
            _fmt_int(r['HGNC 1→1 ATM']),
            _fmt_int(r['HGNC 1→n TDM']),
            _fmt_int(r['HGNC 1→n ATM']),
            _fmt_int(r['Ensembl 1→1']),
            _fmt_int(r['Ensembl 1→n']),
            _fmt_int(r['1→0']),
            _fmt_float(r['Performance (it/s) Ensembl IDs']),
            _fmt_float(r['Performance (it/s) HGNC Symbols']),
        ]
        lines.append(' & '.join(row) + r' \\')

    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    lines.append(r'\end{table*}')

    tex_main = '\n'.join(lines) + '\n'
    atomic_write_text(out_tex_main, tex_main)
    atomic_write_text((ctx.experiment_outputs / 'tables' / out_tex_main.name), tex_main)
    print('Wrote:', out_tex_main)
    print('Wrote:', (ctx.experiment_outputs / 'tables' / out_tex_main.name))

    # Extended alternative (adds drift + collision diagnostics)
    cols_ext = cols_main + [
        'Changed-only 1→1 (Ensembl)',
        'Changed-only 1→n (Ensembl)',
        'n→1 targets within 1→1 (Ensembl)',
        'Unique 1→1 targets (Ensembl)',
        'Unique 1→1 targets (HGNC, incl fallback)',
        'n→1 targets within 1→1 (HGNC, incl fallback)',
    ]

    ext = summary[cols_ext].copy()
    ext_path_csv = HLCA_CACHE / 'hlca_harmonization_extended.csv'
    atomic_write_text(ext_path_csv, ext.to_csv(index=False))
    print('Wrote:', ext_path_csv)

    lines2 = []
    lines2.append(r'\begin{table*}[t]')
    lines2.append(r'\caption{Extended HLCA harmonization diagnostics.}')
    lines2.append(r'\label{tab:hlca_extended}')
    lines2.append(r'\centering')
    lines2.append(r'\footnotesize')
    lines2.append(r'\setlength{\tabcolsep}{3pt}')
    lines2.append(r'\renewcommand{\arraystretch}{1.15}')
    lines2.append(r'\begin{tabular}{@{}l r r r r r r r r r r r r r r@{}}')
    lines2.append(r'\toprule')
    lines2.append(r'Dataset & Input & HGNC 1→1 & HGNC 1→n & Ensembl 1→1 & Ensembl 1→n & 1→0 & Drift 1→1 & Drift 1→n & n→1 Ens & Unique Ens & Unique HGNC & n→1 HGNC & it/s Ens & it/s HGNC \\')
    lines2.append(r'\midrule')

    for _, r in ext.iterrows():
        ds = _latex_escape(r['Dataset'])
        lines2.append(
            ' & '.join(
                [
                    f"\\textit{{{ds}}}",
                    _fmt_int(r['Input IDs']),
                    _fmt_int(r['HGNC 1→1 TDM'] + r['HGNC 1→1 ATM']),
                    _fmt_int(r['HGNC 1→n TDM'] + r['HGNC 1→n ATM']),
                    _fmt_int(r['Ensembl 1→1']),
                    _fmt_int(r['Ensembl 1→n']),
                    _fmt_int(r['1→0']),
                    _fmt_int(r['Changed-only 1→1 (Ensembl)']),
                    _fmt_int(r['Changed-only 1→n (Ensembl)']),
                    _fmt_int(r['n→1 targets within 1→1 (Ensembl)']),
                    _fmt_int(r['Unique 1→1 targets (Ensembl)']),
                    _fmt_int(r['Unique 1→1 targets (HGNC, incl fallback)']),
                    _fmt_int(r['n→1 targets within 1→1 (HGNC, incl fallback)']),
                    _fmt_float(r['Performance (it/s) Ensembl IDs']),
                    _fmt_float(r['Performance (it/s) HGNC Symbols']),
                ]
            )
            + r' \\'
        )

    lines2.append(r'\bottomrule')
    lines2.append(r'\end{tabular}')
    lines2.append(r'\end{table*}')

    tex_ext = '\n'.join(lines2) + '\n'
    atomic_write_text(out_tex_ext, tex_ext)
    atomic_write_text((ctx.experiment_outputs / 'tables' / out_tex_ext.name), tex_ext)
    print('Wrote:', out_tex_ext)
    print('Wrote:', (ctx.experiment_outputs / 'tables' / out_tex_ext.name))

    # Fractions table (alternative; easier to compare across datasets)
    frac = summary.copy()
    denom = frac['Input IDs'].replace(0, pd.NA)
    frac['HGNC 1→1 (frac)'] = (frac['HGNC 1→1 TDM'] + frac['HGNC 1→1 ATM']) / denom
    frac['HGNC 1→n (frac)'] = (frac['HGNC 1→n TDM'] + frac['HGNC 1→n ATM']) / denom
    frac['Ensembl 1→1 (frac)'] = frac['Ensembl 1→1'] / denom
    frac['Ensembl 1→n (frac)'] = frac['Ensembl 1→n'] / denom
    frac['1→0 (frac)'] = frac['1→0'] / denom

    frac_cols = [
        'Dataset',
        'Input IDs',
        'HGNC 1→1 (frac)',
        'HGNC 1→n (frac)',
        'Ensembl 1→1 (frac)',
        'Ensembl 1→n (frac)',
        '1→0 (frac)',
    ]

    frac_df = frac[frac_cols].copy()
    frac_path_csv = HLCA_CACHE / 'hlca_harmonization_fractions.csv'
    atomic_write_text(frac_path_csv, frac_df.to_csv(index=False))
    print('Wrote:', frac_path_csv)

    out_tex_frac = MANUSCRIPT_TABLES / 'hlca_harmonization_fractions.tex'
    lines3 = []
    lines3.append(r'\begin{table}[t]')
    lines3.append(r'\caption{HLCA harmonization outcome fractions (alternative view of Table~\ref{tab:hlca}).}')
    lines3.append(r'\label{tab:hlca_fractions}')
    lines3.append(r'\centering')
    lines3.append(r'\footnotesize')
    lines3.append(r'\setlength{\tabcolsep}{3pt}')
    lines3.append(r'\renewcommand{\arraystretch}{1.15}')
    lines3.append(r'\begin{tabular}{@{}l r r r r r r@{}}')
    lines3.append(r'\toprule')
    lines3.append(r'Dataset & Input & HGNC 1→1 & HGNC 1→n & Ensembl 1→1 & Ensembl 1→n & 1→0 \\')
    lines3.append(r'\midrule')
    for _, r in frac_df.iterrows():
        ds = _latex_escape(r['Dataset'])
        lines3.append(
            ' & '.join(
                [
                    f"\\textit{{{ds}}}",
                    _fmt_int(r['Input IDs']),
                    _fmt_float(r['HGNC 1→1 (frac)'], digits=2),
                    _fmt_float(r['HGNC 1→n (frac)'], digits=2),
                    _fmt_float(r['Ensembl 1→1 (frac)'], digits=2),
                    _fmt_float(r['Ensembl 1→n (frac)'], digits=2),
                    _fmt_float(r['1→0 (frac)'], digits=2),
                ]
            )
            + r' \\\\ '
        )
    lines3.append(r'\bottomrule')
    lines3.append(r'\end{tabular}')
    lines3.append(r'\end{table}')
    tex_frac = '\n'.join(lines3) + '\n'
    atomic_write_text(out_tex_frac, tex_frac)
    atomic_write_text((ctx.experiment_outputs / 'tables' / out_tex_frac.name), tex_frac)
    print('Wrote:', out_tex_frac)
    print('Wrote:', (ctx.experiment_outputs / 'tables' / out_tex_frac.name))

    # Feature-space summary table (union vs intersection; target comparison)
    fs_path = HLCA_CACHE / 'hlca_feature_space_summary.csv'
    if fs_path.exists():
        fs = pd.read_csv(fs_path)
        out_tex_fs = MANUSCRIPT_TABLES / 'hlca_feature_space_summary.tex'
        lines4 = []
        lines4.append(r'\begin{table}[t]')
        lines4.append(r'\caption{HLCA feature-space summary for stable 1→1 targets (union vs intersection across datasets).}')
        lines4.append(r'\label{tab:hlca_feature_space}')
        lines4.append(r'\centering')
        lines4.append(r'\footnotesize')
        lines4.append(r'\begin{tabular}{@{}l r r r@{}}')
        lines4.append(r'\toprule')
        lines4.append(r'Target & Datasets & Union genes & Intersection genes \\')
        lines4.append(r'\midrule')
        for _, r in fs.iterrows():
            lines4.append(
                ' & '.join(
                    [
                        _latex_escape(r['target']),
                        _fmt_int(r['n_datasets']),
                        _fmt_int(r['union_genes']),
                        _fmt_int(r['intersection_genes']),
                    ]
                )
                + r' \\\\ '
            )
        lines4.append(r'\bottomrule')
        lines4.append(r'\end{tabular}')
        lines4.append(r'\end{table}')
        tex_fs = '\n'.join(lines4) + '\n'
        atomic_write_text(out_tex_fs, tex_fs)
        atomic_write_text((ctx.experiment_outputs / 'tables' / out_tex_fs.name), tex_fs)
        print('Wrote:', out_tex_fs)
        print('Wrote:', (ctx.experiment_outputs / 'tables' / out_tex_fs.name))


In [ ]:
# -------------------- Manuscript-style plots --------------------

import numpy as np

if summary.empty:
    print('No summary available; skipping plots.')
else:
    df = summary.set_index('Dataset')

    # 1) Outcome profile per dataset (HGNC target)
    frac_hgnc = df[[
        'HGNC 1→1 TDM', 'HGNC 1→1 ATM', 'HGNC 1→n TDM', 'HGNC 1→n ATM', '1→0'
    ]].div(df['Input IDs'], axis=0)

    frac_hgnc = frac_hgnc.sort_values('1→0', ascending=False)

    colors = {
        'HGNC 1→1 TDM': MANUSCRIPT_COLORS['HGNC 1→1 TDM'],
        'HGNC 1→1 ATM': MANUSCRIPT_COLORS['HGNC 1→1 ATM'],
        'HGNC 1→n TDM': MANUSCRIPT_COLORS['HGNC 1→n TDM'],
        'HGNC 1→n ATM': MANUSCRIPT_COLORS['HGNC 1→n ATM'],
        '1→0': MANUSCRIPT_COLORS['1→0'],
    }

    fig, ax = plt.subplots(1, 1, figsize=(10, max(3.5, 0.25 * len(frac_hgnc))))

    left = None
    for col in frac_hgnc.columns:
        ax.barh(frac_hgnc.index, frac_hgnc[col], left=left, label=col, color=colors.get(col))
        left = frac_hgnc[col] if left is None else (left + frac_hgnc[col])

    ax.set_xlabel('Fraction of input IDs')
    ax.set_ylabel('HLCA dataset')
    ax.set_xlim(0, 1)
    ax.legend(loc='lower right', frameon=True)
    ax.set_title('HLCA identifier harmonization outcomes (target: HGNC symbols)')

    fig.tight_layout()
    written = save_figure(fig, 'fig_hlca_outcomes_hgnc.pdf', ctx, formats=('pdf',))
    print('Saved:', written['pdf'])

    # 2) Target comparison: ambiguity + failure fraction (HGNC vs Ensembl)
    frac_targets = pd.DataFrame(
        {
            'HGNC 1→n (total)': (df['HGNC 1→n TDM'] + df['HGNC 1→n ATM']) / df['Input IDs'],
            'Ensembl 1→n': df['Ensembl 1→n'] / df['Input IDs'],
            '1→0': df['1→0'] / df['Input IDs'],
        }
    ).sort_values('1→0', ascending=False)

    fig2, ax2 = plt.subplots(1, 1, figsize=(10, max(3.5, 0.25 * len(frac_targets))))
    frac_targets[['HGNC 1→n (total)', 'Ensembl 1→n', '1→0']].plot(
        kind='barh',
        ax=ax2,
        color=['#DD8452', '#4C72B0', '#C44E52'],
    )
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Fraction of input IDs')
    ax2.set_ylabel('HLCA dataset')
    ax2.set_title('Ambiguity and failure rates by target namespace')
    ax2.legend(loc='lower right', frameon=True)
    fig2.tight_layout()

    written2 = save_figure(fig2, 'fig_hlca_target_compare.pdf', ctx, formats=('pdf',))
    print('Saved:', written2['pdf'])

    # 3) Cross-dataset overlap distribution (Ensembl 1→1 only)
    overlap_path = HLCA_CACHE / 'hlca_gene_overlap_distribution.csv'
    if overlap_path.exists():
        overlap_df = pd.read_csv(overlap_path)

        fig3, ax3 = plt.subplots(1, 1, figsize=(6.5, 4))
        ax3.bar(overlap_df['n_datasets_present'], overlap_df['n_genes'], color='#4C72B0')
        ax3.set_xlabel('# datasets a gene is present in (after 1→1 mapping)')
        ax3.set_ylabel('# genes')
        ax3.set_title('HLCA shared feature-space structure (Ensembl 1→1 only)')
        fig3.tight_layout()

        written3 = save_figure(fig3, 'fig_hlca_gene_overlap.pdf', ctx, formats=('pdf',))
        print('Saved:', written3['pdf'])
    else:
        print('No overlap CSV found; skipping gene-overlap plot:', overlap_path)

    # 4) Throughput scaling (marketing: atlas-scale practicality)
    perf = summary[['Dataset', 'Input IDs', 'Performance (it/s) Ensembl IDs', 'Performance (it/s) HGNC Symbols']].copy()
    perf = perf.dropna()

    if not perf.empty:
        fig4, ax4 = plt.subplots(1, 1, figsize=(6.8, 4.2))
        ax4.scatter(
            perf['Input IDs'],
            perf['Performance (it/s) Ensembl IDs'],
            label='Target: Ensembl gene IDs',
            color=MANUSCRIPT_COLORS['1→1'],
            s=35,
            alpha=0.9,
        )
        ax4.scatter(
            perf['Input IDs'],
            perf['Performance (it/s) HGNC Symbols'],
            label='Target: HGNC symbols',
            color=MANUSCRIPT_COLORS['1→n'],
            s=35,
            alpha=0.9,
        )
        ax4.set_xscale('log')
        ax4.set_yscale('log')
        ax4.set_xlabel('# input identifiers (log scale)')
        ax4.set_ylabel('Throughput (it/s; log scale)')
        ax4.set_title('HLCA harmonization throughput scales across datasets')
        ax4.legend(loc='best', frameon=True)
        fig4.tight_layout()

        written4 = save_figure(fig4, 'fig_hlca_throughput_scaling.pdf', ctx, formats=('pdf',))
        print('Saved:', written4['pdf'])
    else:
        print('No throughput timings available; skipping throughput scaling plot.')

    # 5) Shared feature-space structure: pairwise Jaccard heatmap (Ensembl 1→1 only)
    jaccard_path = HLCA_CACHE / 'hlca_jaccard_matrix.csv'
    if jaccard_path.exists():
        jacc = pd.read_csv(jaccard_path, index_col=0)

        fig5, ax5 = plt.subplots(1, 1, figsize=(8.5, 7.5))
        if sns is not None:
            sns.heatmap(jacc, ax=ax5, cmap='Blues', vmin=0, vmax=1, square=True, cbar_kws={'label': 'Jaccard'} )
        else:
            im = ax5.imshow(jacc.values, cmap='Blues', vmin=0, vmax=1)
            fig5.colorbar(im, ax=ax5, label='Jaccard')
            ax5.set_xticks(range(len(jacc.columns)))
            ax5.set_yticks(range(len(jacc.index)))
            ax5.set_xticklabels(jacc.columns, rotation=90)
            ax5.set_yticklabels(jacc.index)

        ax5.set_title('HLCA pairwise overlap after 1→1 harmonization (Ensembl backbone)')
        ax5.set_xlabel('Dataset')
        ax5.set_ylabel('Dataset')
        fig5.tight_layout()

        written5 = save_figure(fig5, 'fig_hlca_jaccard_heatmap.pdf', ctx, formats=('pdf',))
        print('Saved:', written5['pdf'])
    else:
        print('No Jaccard matrix found; skipping heatmap:', jaccard_path)

    # 6) Shared feature-space structure: pairwise Jaccard heatmap (HGNC 1→1 incl fallback)
    jaccard_path_hgnc = HLCA_CACHE / 'hlca_hgnc_jaccard_matrix.csv'
    if jaccard_path_hgnc.exists():
        jacc_hgnc = pd.read_csv(jaccard_path_hgnc, index_col=0)

        fig6, ax6 = plt.subplots(1, 1, figsize=(8.5, 7.5))
        if sns is not None:
            sns.heatmap(
                jacc_hgnc,
                ax=ax6,
                cmap='Blues',
                vmin=0,
                vmax=1,
                square=True,
                cbar_kws={'label': 'Jaccard'},
            )
        else:
            im = ax6.imshow(jacc_hgnc.values, cmap='Blues', vmin=0, vmax=1)
            fig6.colorbar(im, ax=ax6, label='Jaccard')
            ax6.set_xticks(range(len(jacc_hgnc.columns)))
            ax6.set_yticks(range(len(jacc_hgnc.index)))
            ax6.set_xticklabels(jacc_hgnc.columns, rotation=90)
            ax6.set_yticklabels(jacc_hgnc.index)

        ax6.set_title('HLCA pairwise overlap after 1→1 harmonization (target: HGNC symbols)')
        ax6.set_xlabel('Dataset')
        ax6.set_ylabel('Dataset')
        fig6.tight_layout()

        written6 = save_figure(fig6, 'fig_hlca_jaccard_heatmap_hgnc.pdf', ctx, formats=('pdf',))
        print('Saved:', written6['pdf'])
    else:
        print('No HGNC Jaccard matrix found; skipping heatmap:', jaccard_path_hgnc)

    # 7) Stable 1→1 feature-space size per dataset (Ensembl vs HGNC)
    if (
        'Unique 1→1 targets (Ensembl)' in summary.columns
        and 'Unique 1→1 targets (HGNC, incl fallback)' in summary.columns
    ):
        comp = (
            summary.set_index('Dataset')[
                ['Unique 1→1 targets (Ensembl)', 'Unique 1→1 targets (HGNC, incl fallback)']
            ]
            .sort_values('Unique 1→1 targets (Ensembl)', ascending=False)
        )

        fig7, ax7 = plt.subplots(1, 1, figsize=(10, max(3.5, 0.25 * len(comp))))
        comp.plot(
            kind='barh',
            ax=ax7,
            color=[MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
        )
        ax7.set_xlabel('# unique 1→1 targets')
        ax7.set_ylabel('HLCA dataset')
        ax7.set_title('Stable 1→1 feature-space size by target namespace')
        ax7.legend(['Ensembl', 'HGNC (incl fallback)'], loc='lower right', frameon=True)
        fig7.tight_layout()

        written7 = save_figure(fig7, 'fig_hlca_unique_targets_compare.pdf', ctx, formats=('pdf',))
        print('Saved:', written7['pdf'])
    else:
        print('Unique-target columns missing; skipping unique-target comparison plot.')

    # 8) Global feature-space summary (union vs intersection; target comparison)
    fs_path = HLCA_CACHE / 'hlca_feature_space_summary.csv'
    if fs_path.exists():
        fs = pd.read_csv(fs_path)
        melt = fs.melt(
            id_vars=['target', 'n_datasets'],
            value_vars=['union_genes', 'intersection_genes'],
            var_name='metric',
            value_name='n_genes',
        )

        fig8, ax8 = plt.subplots(1, 1, figsize=(7.5, 3.8))
        if sns is not None:
            sns.barplot(data=melt, x='target', y='n_genes', hue='metric', ax=ax8)
        else:
            pivot = melt.pivot(index='target', columns='metric', values='n_genes')
            pivot.plot(kind='bar', ax=ax8)

        ax8.set_ylabel('# genes')
        ax8.set_xlabel('Target definition')
        ax8.set_title('HLCA stable feature-space: union vs intersection (1→1 only)')
        ax8.tick_params(axis='x', rotation=15)
        ax8.legend(loc='best', frameon=True, title='Metric')
        fig8.tight_layout()

        written8 = save_figure(fig8, 'fig_hlca_feature_space_union_intersection.pdf', ctx, formats=('pdf',))
        print('Saved:', written8['pdf'])
    else:
        print('No feature-space summary found; skipping plot:', fs_path)

    # 9) Optional combined marketing figure (multi-panel)
    try:
        fig9, axes9 = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)
        axA, axB, axC, axD = axes9.ravel()

        # Panel A: HGNC outcome profile (re-use frac_hgnc from earlier)
        frac_hgnc[['HGNC 1→1 TDM', 'HGNC 1→1 ATM', 'HGNC 1→n TDM', 'HGNC 1→n ATM', '1→0']].plot(
            kind='barh',
            stacked=True,
            ax=axA,
            color=[
                MANUSCRIPT_COLORS['HGNC 1→1 TDM'],
                MANUSCRIPT_COLORS['HGNC 1→1 ATM'],
                MANUSCRIPT_COLORS['HGNC 1→n TDM'],
                MANUSCRIPT_COLORS['HGNC 1→n ATM'],
                MANUSCRIPT_COLORS['1→0'],
            ],
        )
        axA.set_xlim(0, 1)
        axA.set_title('A) Outcomes per dataset (HGNC target)')
        axA.set_xlabel('Fraction of input IDs')
        axA.set_ylabel('')
        axA.legend(loc='lower right', frameon=True)

        # Panel B: Target comparison (HGNC vs Ensembl)
        frac_targets[['HGNC 1→n (total)', 'Ensembl 1→n', '1→0']].plot(kind='barh', ax=axB)
        axB.set_xlim(0, 1)
        axB.set_title('B) Ambiguity + failure by target')
        axB.set_xlabel('Fraction of input IDs')
        axB.set_ylabel('')

        # Panel C: Unique targets compare (if computed)
        if 'comp' in locals():
            comp.plot(kind='barh', ax=axC, color=[MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']])
            axC.set_title('C) Stable 1→1 feature-space size')
            axC.set_xlabel('# unique 1→1 targets')
            axC.set_ylabel('')
        else:
            axC.axis('off')

        # Panel D: Global union vs intersection (if available)
        if 'fs' in locals() and not fs.empty:
            melt = fs.melt(
                id_vars=['target', 'n_datasets'],
                value_vars=['union_genes', 'intersection_genes'],
                var_name='metric',
                value_name='n_genes',
            )
            if sns is not None:
                sns.barplot(data=melt, x='target', y='n_genes', hue='metric', ax=axD)
            else:
                melt.pivot(index='target', columns='metric', values='n_genes').plot(kind='bar', ax=axD)
            axD.set_title('D) Union vs intersection (1→1 only)')
            axD.set_xlabel('')
            axD.set_ylabel('# genes')
            axD.tick_params(axis='x', rotation=15)
            axD.legend(loc='best', frameon=True, title='Metric')
        else:
            axD.axis('off')

        written9 = save_figure(fig9, 'fig_hlca_marketing_suite.pdf', ctx, formats=('pdf',))
        print('Saved:', written9['pdf'])
    except Exception as e:  # noqa: S110
        print('Skipping combined marketing figure due to error:', repr(e))


# Marketing extension: compare cross-dataset similarity (Ensembl vs HGNC)

Harmonization is not just about per-dataset outcome profiles; it is also about the **structure of the shared feature space**.

This section compares *pairwise dataset similarity* after 1→1 mapping under two target definitions:

- Ensembl backbone (strict 1→1 only)
- HGNC symbols (1→1 including alternative-target fallbacks)

It produces a manuscript-ready multi-panel figure with:

- two heatmaps (Ensembl vs HGNC)
- a difference heatmap (HGNC − Ensembl)
- a scatter plot summarizing off-diagonal similarity


In [ ]:
from experiments_utils import atomic_write_dataframe_csv, label_panels  # noqa: E402
from plotting_utils import heatmap  # noqa: E402

ens_path = HLCA_CACHE / 'hlca_jaccard_matrix.csv'
hgnc_path = HLCA_CACHE / 'hlca_hgnc_jaccard_matrix.csv'

if (not ens_path.exists()) or (not hgnc_path.exists()):
    print('Missing one of the Jaccard matrices; run the table-building cell first.')
    print('Expected:', ens_path)
    print('Expected:', hgnc_path)
else:
    ens = pd.read_csv(ens_path, index_col=0)
    hgnc = pd.read_csv(hgnc_path, index_col=0)

    # Align to common dataset order
    common = [s for s in ens.index if s in set(hgnc.index)]
    ens = ens.loc[common, common]
    hgnc = hgnc.loc[common, common]
    delta = hgnc - ens

    def _mean_offdiag(mat: pd.DataFrame) -> float:
        a = mat.values.astype(float)
        if a.size == 0:
            return float('nan')
        m = ~np.eye(a.shape[0], dtype=bool)
        return float(np.nanmean(a[m]))

    summary = pd.DataFrame(
        [
            {'target': 'Ensembl (1→1 only)', 'mean_offdiag_jaccard': _mean_offdiag(ens)},
            {'target': 'HGNC (1→1 incl fallback)', 'mean_offdiag_jaccard': _mean_offdiag(hgnc)},
        ]
    )
    summary['delta_vs_ensembl'] = summary['mean_offdiag_jaccard'] - float(summary.iloc[0]['mean_offdiag_jaccard'])

    out_sum = MANUSCRIPT_TABLES / 'hlca_jaccard_summary.csv'
    atomic_write_dataframe_csv(summary, out_sum, index=False)
    atomic_write_dataframe_csv(summary, (ctx.experiment_outputs / 'tables' / out_sum.name), index=False)
    print('Wrote:', out_sum)
    display(summary)

    fig, axes = plt.subplots(2, 2, figsize=(14, 11), constrained_layout=True)
    axA, axB, axC, axD = axes.ravel()

    heatmap(axA, ens, title='Ensembl target — pairwise dataset similarity (Jaccard)', cmap='Blues', vmin=0, vmax=1, square=True)
    heatmap(axB, hgnc, title='HGNC target — pairwise dataset similarity (Jaccard)', cmap='Blues', vmin=0, vmax=1, square=True)

    vmax = float(np.nanmax(np.abs(delta.values))) if delta.size else 0.0
    vmax = max(vmax, 0.05)
    heatmap(
        axC,
        delta,
        title='HGNC − Ensembl (difference in Jaccard)',
        cmap='RdBu_r',
        vmin=-vmax,
        vmax=vmax,
        center=0.0,
        square=True,
        cbar=True,
        cbar_label='Δ',
    )

    a = ens.values.astype(float)
    b = hgnc.values.astype(float)
    mask = ~np.eye(a.shape[0], dtype=bool)
    x = a[mask]
    y = b[mask]
    axD.scatter(x, y, s=10, alpha=0.35, color=MANUSCRIPT_COLORS['IDTrack'])
    axD.plot([0, 1], [0, 1], '--', lw=1.0, color=MANUSCRIPT_COLORS['neutral'])
    axD.set_xlim(0, 1)
    axD.set_ylim(0, 1)
    axD.set_xlabel('Ensembl Jaccard')
    axD.set_ylabel('HGNC Jaccard')
    axD.set_title('Off-diagonal similarity structure (HGNC vs Ensembl)')

    label_panels(axes.ravel())
    written = save_figure(fig, 'fig_hlca_jaccard_comparison.pdf', ctx, formats=('pdf',))
    print('Saved:', written['pdf'])
